In [1]:
# Import packages
import pandas as pd
import statsmodels.api as sm

In [3]:
# # Load the dataset from your local file path into a pandas DataFrame (You will need to change the directory)
df = pd.read_csv("/Users/emilymoore/Downloads/DS 4002 Project 1/cleaned_housing_market_data.csv")
df['date'] = pd.to_datetime(df['date']) # Convert the 'date' column to datetime format so Python recognizes it as time-series data
df = df.sort_values('date') # Sort the DataFrame chronologically to ensure proper time ordering

# Keep relevant variables
ts = df[['date','price_volatility','uncertainty_score']].dropna()
ts = ts.set_index('date') # Set the date column as the index so the data behaves like a time series

# Create a lagged version of price volatility (Vol_{t-1})
ts['vol_lag1'] = ts['price_volatility'].shift(1) # shift(1) moves the series down one period so each row contains the previous month's volatility
ts = ts.dropna() # Drop the first observation (which becomes NaN due to lagging)

# Add a constant term (intercept α) to the regression model
X = sm.add_constant(ts[['uncertainty_score','vol_lag1']]) # Add a constant term (intercept α) to the regression model
y = ts['price_volatility'] # Define the dependent variable (current price volatility)

# Estimate the OLS regression:
model_ar = sm.OLS(y, X).fit() # Vol_t = α + β1*Uncertainty_t + β2*Vol_{t-1} + ε_t
print(model_ar.summary()) # Print the full regression output including coefficients, R², and significance tests

                            OLS Regression Results                            
Dep. Variable:       price_volatility   R-squared:                       0.326
Model:                            OLS   Adj. R-squared:                  0.057
Method:                 Least Squares   F-statistic:                     1.212
Date:                Sun, 15 Feb 2026   Prob (F-statistic):              0.372
Time:                        09:36:38   Log-Likelihood:                 13.067
No. Observations:                   8   AIC:                            -20.13
Df Residuals:                       5   BIC:                            -19.89
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -0.0119      0.03

/opt/anaconda3/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=8 observations were given.
  return hypotest_fun_in(*args, **kwds)
